# 01 - Coleta, EDA e Pre-processamento

Notebook 1: carrega os arquivos `raw`, monta o dataset final, faz EDA rapida, separa treino/teste e salva os artefatos para o notebook 2.

## 1) Setup

In [ ]:
import re
import unicodedata
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 42
ROOT = Path("..").resolve()
RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"
MODELS_DIR = ROOT / "models"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("RAW:", RAW_DIR)
print("PROCESSED:", PROCESSED_DIR)

## 2) Carga dos arquivos raw usados no fluxo original

In [ ]:
censo = pd.read_csv(RAW_DIR / "censo2022_municipios.csv", dtype={"cod_ibge": str})
censo["cod_ibge"] = censo["cod_ibge"].str.zfill(7)

atlas_raw = pd.read_excel(RAW_DIR / "atlas_brasil_municipios.xlsx")
ibge_ref = pd.read_csv(RAW_DIR / "ibge_municipios_ref.csv", dtype={"cod_ibge": str})
pib_raw = pd.read_csv(RAW_DIR / "pib_municipios_sidra_2021.csv", dtype={"cod_ibge": str})

try:
    areas_raw = pd.read_excel(RAW_DIR / "areas_municipios_2024.xls", dtype={"CD_MUN": str})
except ImportError as exc:
    raise ImportError("Instale xlrd para ler areas_municipios_2024.xls: pip install xlrd") from exc

xl_sinesp = pd.ExcelFile(RAW_DIR / "sinesp_municipios.xlsx")
sinesp_raw = pd.concat([xl_sinesp.parse(sheet) for sheet in xl_sinesp.sheet_names], ignore_index=True)

print("censo:", censo.shape)
print("atlas:", atlas_raw.shape)
print("ibge_ref:", ibge_ref.shape)
print("pib_raw:", pib_raw.shape)
print("areas_raw:", areas_raw.shape)
print("sinesp_raw:", sinesp_raw.shape)

## 3) Tratamento Atlas + chave IBGE

In [ ]:
def normalizar_nome(nome: str) -> str:
    if not isinstance(nome, str):
        return ""
    nfkd = unicodedata.normalize("NFKD", nome)
    sem_acento = "".join(c for c in nfkd if not unicodedata.combining(c))
    return re.sub(r"\s+", " ", sem_acento).strip().lower()

atlas_raw.columns = [
    "territorialidade",
    "gini",
    "idhm",
    "idhm_renda",
    "idhm_longevidade",
    "idhm_educacao",
    "perc_pobres",
]
atlas_raw = atlas_raw[atlas_raw["territorialidade"] != "Brasil"].copy()
atlas_parts = atlas_raw["territorialidade"].str.extract(r"^(.*)\s+\((\w{2})\)$")
atlas_raw["nome_mun"] = atlas_parts[0]
atlas_raw["uf"] = atlas_parts[1]
atlas_raw = atlas_raw[atlas_raw["nome_mun"].notna() & atlas_raw["uf"].notna()].copy()
atlas_raw["nome_norm"] = atlas_raw["nome_mun"].map(normalizar_nome)

if "nome_norm" not in ibge_ref.columns:
    ibge_ref["nome_norm"] = ibge_ref["nome_ibge"].map(normalizar_nome)

atlas = atlas_raw.merge(
    ibge_ref[["cod_ibge", "nome_norm", "uf"]],
    on=["nome_norm", "uf"],
    how="left",
)
atlas = atlas[["cod_ibge", "gini", "idhm", "idhm_renda", "idhm_longevidade", "idhm_educacao", "perc_pobres"]]

for col in ["gini", "idhm", "idhm_renda", "idhm_longevidade", "idhm_educacao", "perc_pobres"]:
    atlas[col] = pd.to_numeric(atlas[col], errors="coerce")

atlas_sem_codigo = int(atlas["cod_ibge"].isna().sum())
print("atlas sem cod_ibge:", atlas_sem_codigo)
if atlas_sem_codigo > 600:
    raise ValueError("Join Atlas x IBGE com baixa cobertura. Verifique normalizacao de nomes e UF.")
atlas.head()

## 4) PIB, areas e variavel alvo

In [ ]:
pib = pib_raw.merge(censo[["cod_ibge", "pop_total"]], on="cod_ibge", how="left")
pib["pib_total_mil"] = pd.to_numeric(pib["pib_total_mil"], errors="coerce")
pib["pib_per_capita"] = (pib["pib_total_mil"] * 1_000) / pib["pop_total"].replace(0, np.nan)
pib = pib[["cod_ibge", "pib_per_capita"]]

areas = areas_raw[["CD_MUN", "AR_MUN_2024"]].rename(columns={"CD_MUN": "cod_ibge", "AR_MUN_2024": "area_km2"})
areas["cod_ibge"] = areas["cod_ibge"].astype(str).str.zfill(7)
areas["area_km2"] = pd.to_numeric(areas["area_km2"], errors="coerce")

sinesp_raw = sinesp_raw.rename(columns={"C\u00f3d_IBGE": "cod_ibge", "M\u00eas/Ano": "mes_ano", "V\u00edtimas": "vitimas"})
sinesp_raw["cod_ibge"] = sinesp_raw["cod_ibge"].astype(str).str.replace(".0", "", regex=False).str.zfill(7)
sinesp_raw["mes_ano"] = pd.to_datetime(sinesp_raw["mes_ano"], errors="coerce")
sinesp_raw["vitimas"] = pd.to_numeric(sinesp_raw["vitimas"], errors="coerce")

sinesp_2022 = (
    sinesp_raw[sinesp_raw["mes_ano"].dt.year == 2022]
    .groupby("cod_ibge", as_index=False)["vitimas"]
    .sum()
    .rename(columns={"vitimas": "homicidios_2022"})
)

base = censo[["cod_ibge", "pop_total"]].copy()
base = base.merge(sinesp_2022, on="cod_ibge", how="left")
base["homicidios_2022"] = base["homicidios_2022"].fillna(0)
base["taxa_homicidios"] = (base["homicidios_2022"] / base["pop_total"].replace(0, np.nan)) * 100_000

limiar = base["taxa_homicidios"].median()
base["alta_violencia"] = (base["taxa_homicidios"] > limiar).astype(int)

print("limiar taxa_homicidios:", round(limiar, 4))
print(base["alta_violencia"].value_counts().to_dict())

## 5) Montagem do dataset final

In [ ]:
df = base[["cod_ibge", "pop_total", "alta_violencia", "taxa_homicidios"]].copy()
df = df.merge(censo.drop(columns=["pop_total"]), on="cod_ibge", how="left")
df = df.merge(atlas, on="cod_ibge", how="left")
df = df.merge(pib, on="cod_ibge", how="left")
df = df.merge(areas, on="cod_ibge", how="left")
df["densidade_demografica"] = df["pop_total"] / df["area_km2"].replace(0, np.nan)

FEATURES = [
    "pop_total",
    "pop_urbana_pct",
    "perc_jovens_15_29",
    "renda_per_capita",
    "taxa_desemprego",
    "taxa_analfabetismo_15",
    "perc_esgoto_adequado",
    "gini",
    "idhm",
    "idhm_renda",
    "idhm_longevidade",
    "idhm_educacao",
    "perc_pobres",
    "pib_per_capita",
    "densidade_demografica",
]

df = df[df["pop_total"] > 0].copy()
for col in FEATURES:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].median())

dataset = (
    df[["cod_ibge"] + FEATURES + ["taxa_homicidios", "alta_violencia"]]
    .drop_duplicates(subset="cod_ibge")
    .sort_values("cod_ibge")
    .reset_index(drop=True)
)

print("shape:", dataset.shape)
print("nulos totais:", int(dataset.isna().sum().sum()))
if dataset[FEATURES + ["taxa_homicidios", "alta_violencia"]].isna().any().any():
    raise ValueError("Dataset final contem NaN nas colunas usadas no modelo.")
dataset.head()

## 6) EDA rapida

In [ ]:
print("municipios:", len(dataset))
print("classes:", dataset["alta_violencia"].value_counts().to_dict())

corr_target = (
    dataset[FEATURES + ["alta_violencia"]]
    .corr(numeric_only=True)["alta_violencia"]
    .drop("alta_violencia")
    .sort_values(key=lambda s: s.abs(), ascending=False)
)
print("top correlacoes:")
print(corr_target.head(8).round(4).to_string())

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
dataset["alta_violencia"].value_counts().sort_index().plot(kind="bar", ax=ax[0], color=["#4c78a8", "#f58518"])
ax[0].set_title("Distribuicao de y")
ax[0].set_xlabel("alta_violencia")
ax[0].set_ylabel("quantidade")

dataset["taxa_homicidios"].plot(kind="hist", bins=40, ax=ax[1], color="#54a24b")
ax[1].set_title("Taxa de homicidios")
plt.tight_layout()
plt.show()

## 7) X, y, split e padronizacao

In [ ]:
X = dataset[FEATURES].copy()
y = dataset["alta_violencia"].astype(int).copy()

N, p = X.shape
print("N:", N)
print("p:", p)

X_train_df, X_test_df, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y,
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_df)
X_test = scaler.transform(X_test_df)

print("treino:", X_train.shape, y_train.shape)
print("teste:", X_test.shape, y_test.shape)

## 8) Salvamento dos artefatos

In [ ]:
dataset.to_csv(PROCESSED_DIR / "dataset_municipios.csv", index=False)
np.save(PROCESSED_DIR / "X_train.npy", X_train)
np.save(PROCESSED_DIR / "X_test.npy", X_test)
np.save(PROCESSED_DIR / "y_train.npy", y_train.to_numpy())
np.save(PROCESSED_DIR / "y_test.npy", y_test.to_numpy())
(PROCESSED_DIR / "feature_names.txt").write_text("\n".join(FEATURES), encoding="utf-8")
joblib.dump(scaler, MODELS_DIR / "scaler.joblib")

print("arquivos salvos:")
print("- data/processed/dataset_municipios.csv")
print("- data/processed/X_train.npy")
print("- data/processed/X_test.npy")
print("- data/processed/y_train.npy")
print("- data/processed/y_test.npy")
print("- data/processed/feature_names.txt")
print("- models/scaler.joblib")